# White-Box Profit-Oriented Forecast Attack in Two-Settlement Markets

This notebook is a from-scratch, reproducible PES Letters case study. It uses only the public [RTS-GMLC](https://github.com/GridMod/RTS-GMLC) benchmark and does **not** reuse the earlier IEEE 33-bus/SPGA/CBM experiment.

**Central hypothesis.** A rational attacker maximizes private settlement profit rather than forecasting error or total system cost. Such an attack can redistribute market surplus while leaving conventional anomaly indicators nearly unchanged.

The public 73-bus system is aggregated along its three published market areas. All conventional generating units, public marginal-cost inputs, ramp rates, cross-area branch ratings, and DA/RT wind-load trajectories are retained. This aggregation makes repeated attack-market simulations feasible without a commercial solver.

## Threat model and single objective

The attacker corrupts the area-3 day-ahead wind forecast within

$$\left|\delta_t\right|\leq\epsilon\overline W_3,$$

and holds a fixed virtual position $q_t$ learned from the first 30 days' historical DA--RT spread. The test-period objective is only

$$\max_{\boldsymbol\delta}\;\Pi_A(\boldsymbol\delta)
=\sum_t q_t\left[\lambda_{3,t}^{\mathrm{DA}}(\boldsymbol\delta)
-\lambda_{3,t}^{\mathrm{RT}}(\boldsymbol\delta)\right].$$

The position is fixed before the seven test days; realized test prices are never used to choose its direction. The proposed attack is compared with an equal-budget MSE attack and a two-stage system-cost attack.

## 1. Environment and public-data acquisition

The code below automatically clones the official repository when raw data are absent. The current artifact also contains the processed 60-day hourly file used for the reported results.

In [ ]:
from pathlib import Path
import subprocess
PROJECT = Path.cwd()
if not (PROJECT/'RTS-GMLC'/'RTS_Data').exists():
    subprocess.run(['git','clone','--depth','1','--filter=blob:none','--sparse',
                    'https://github.com/GridMod/RTS-GMLC.git',str(PROJECT/'RTS-GMLC')],check=True)
    subprocess.run(['git','-C',str(PROJECT/'RTS-GMLC'),'sparse-checkout','set',
                    'RTS_Data/SourceData','RTS_Data/timeseries_data_files/WIND',
                    'RTS_Data/timeseries_data_files/Load'],check=True)


## 2. Data processing and market parameters

In [ ]:
"""White-box profit-oriented forecast attack on a public RTS-GMLC market case.

The model is intentionally compact enough for a PES Letters reproducibility
notebook: the public 73-bus RTS-GMLC system is aggregated to its three market
areas while preserving every thermal unit, published marginal-cost inputs,
inter-area transfer limits, ramp rates, and public DA/RT wind-load profiles.
"""
from __future__ import annotations

from dataclasses import dataclass
from functools import lru_cache
from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy.optimize import linprog

ROOT = Path.cwd()
RAW = ROOT / "RTS-GMLC" / "RTS_Data"
DATA = ROOT / "data"
RESULTS = ROOT / "results"
FIGURES = ROOT / "figures"
for _p in (DATA, RESULTS, FIGURES):
    _p.mkdir(parents=True, exist_ok=True)

AREAS = np.array([1, 2, 3])
LINKS = [(1, 2), (1, 3), (2, 3)]


def build_public_dataset(days: int = 60) -> pd.DataFrame:
    """Aggregate official 5-min RT profiles to hourly observations."""
    da_w = pd.read_csv(RAW / "timeseries_data_files/WIND/DAY_AHEAD_wind.csv")
    rt_w = pd.read_csv(RAW / "timeseries_data_files/WIND/REAL_TIME_wind.csv")
    da_l = pd.read_csv(RAW / "timeseries_data_files/Load/DAY_AHEAD_regional_Load.csv")
    rt_l = pd.read_csv(RAW / "timeseries_data_files/Load/REAL_TIME_regional_Load.csv")

    def stamp(df, rt=False):
        base = pd.to_datetime(dict(year=df.Year, month=df.Month, day=df.Day))
        if rt:
            return base + pd.to_timedelta((df.Period - 1) * 5, unit="min")
        return base + pd.to_timedelta(df.Period - 1, unit="h")

    da_w["ts"] = stamp(da_w)
    da_l["ts"] = stamp(da_l)
    rt_w["ts"] = stamp(rt_w, True)
    rt_l["ts"] = stamp(rt_l, True)
    cutoff = da_w.ts.min() + pd.Timedelta(days=days)
    da_w, da_l = da_w[da_w.ts < cutoff], da_l[da_l.ts < cutoff]
    rt_w, rt_l = rt_w[rt_w.ts < cutoff], rt_l[rt_l.ts < cutoff]

    # Wind buses 122 -> area 1; 303/309/317 -> area 3.
    out = pd.DataFrame({"timestamp": da_w.ts})
    out["wind_da_1"] = da_w["122_WIND_1"].to_numpy()
    out["wind_da_2"] = 0.0
    out["wind_da_3"] = da_w[["303_WIND_1", "309_WIND_1", "317_WIND_1"]].sum(axis=1).to_numpy()
    for a in AREAS:
        out[f"load_da_{a}"] = da_l[str(a)].to_numpy()

    rt_w_h = rt_w.set_index("ts").resample("1h").mean(numeric_only=True)
    rt_l_h = rt_l.set_index("ts").resample("1h").mean(numeric_only=True)
    out["wind_rt_1"] = rt_w_h["122_WIND_1"].reindex(out.timestamp).to_numpy()
    out["wind_rt_2"] = 0.0
    out["wind_rt_3"] = rt_w_h[["303_WIND_1", "309_WIND_1", "317_WIND_1"]].sum(axis=1).reindex(out.timestamp).to_numpy()
    for a in AREAS:
        out[f"load_rt_{a}"] = rt_l_h[str(a)].reindex(out.timestamp).to_numpy()
    out.to_csv(DATA / "public_market_60d.csv", index=False)
    return out


@dataclass
class MarketData:
    gen_area: np.ndarray
    pmax: np.ndarray
    ramp: np.ndarray
    cost: np.ndarray
    link_cap: np.ndarray
    wind_cap: np.ndarray


def load_market_data() -> MarketData:
    bus = pd.read_csv(RAW / "SourceData/bus.csv").set_index("Bus ID")
    branch = pd.read_csv(RAW / "SourceData/branch.csv")
    gen = pd.read_csv(RAW / "SourceData/gen.csv")
    conventional = gen[gen.Fuel.isin(["Coal", "NG", "Nuclear", "Oil", "Hydro"])].copy()
    conventional["Area"] = conventional["Bus ID"].map(bus.Area)
    heat = pd.to_numeric(conventional["HR_avg_0"], errors="coerce")
    fuel = pd.to_numeric(conventional["Fuel Price $/MMBTU"], errors="coerce")
    vom = pd.to_numeric(conventional.VOM, errors="coerce").fillna(0)
    mc = heat * fuel / 1000 + vom
    # Hydro has no fuel heat-rate; a small opportunity cost retains merit order.
    mc = mc.where(np.isfinite(mc) & (mc > 0), np.where(conventional.Fuel.eq("Hydro"), 5.0, 45.0))
    ramp = pd.to_numeric(conventional["Ramp Rate MW/Min"], errors="coerce").fillna(conventional["PMax MW"] / 60) * 60
    ramp = np.minimum(ramp.to_numpy(float), conventional["PMax MW"].to_numpy(float))

    branch["af"] = branch["From Bus"].map(bus.Area)
    branch["at"] = branch["To Bus"].map(bus.Area)
    caps = []
    for a, b in LINKS:
        m = ((branch.af == a) & (branch.at == b)) | ((branch.af == b) & (branch.at == a))
        caps.append(branch.loc[m, "Cont Rating"].sum())
    # Area aggregation otherwise overstates simultaneous transfer capability.
    caps = 0.65 * np.asarray(caps, float)
    wind_cap = np.array([713.2, 0.0, 1427.7], float)  # public RTS generator PMax by area
    return MarketData(conventional.Area.to_numpy(int), conventional["PMax MW"].to_numpy(float), ramp,
                      np.asarray(mc, float), caps, wind_cap)




## 3. Day-ahead and real-time market clearing

In [ ]:
def _idx(G, T=24):
    nP = G*T; nW = 3*T; nF = 3*T; nS = 3*T
    return slice(0,nP), slice(nP,nP+nW), slice(nP+nW,nP+nW+nF), slice(nP+nW+nF,nP+nW+nF+nS), nP+nW+nF+nS


def clear_da(md: MarketData, load: np.ndarray, wind_avail: np.ndarray,
             use_ramp=True, use_network=True):
    """24-h three-area DA market; returns dispatch, LMP and primal/dual details."""
    G, T = len(md.cost), 24
    pS,wS,fS,sS,N = _idx(G,T)
    c = np.zeros(N); c[pS] = np.repeat(md.cost,T); c[wS] = -0.01; c[sS] = 10000.0
    bounds=[]
    for g in range(G): bounds += [(0,md.pmax[g])]*T
    for a in range(3): bounds += [(0,max(0,wind_avail[a,t])) for t in range(T)]
    fcaps = md.link_cap if use_network else np.full(3,1e5)
    for l in range(3): bounds += [(-fcaps[l],fcaps[l])]*T
    bounds += [(0,None)]*(3*T)
    Aeq=[]; beq=[]
    for a in range(3):
      for t in range(T):
        row=np.zeros(N)
        gs=np.where(md.gen_area==a+1)[0]
        row[gs*T+t]=1
        row[wS.start+a*T+t]=1; row[sS.start+a*T+t]=1
        for l,(u,v) in enumerate(LINKS):
            if a+1==u: row[fS.start+l*T+t]-=1
            if a+1==v: row[fS.start+l*T+t]+=1
        Aeq.append(row); beq.append(load[a,t])
    Aub=[]; bub=[]
    if use_ramp:
      for g in range(G):
       for t in range(1,T):
        r=np.zeros(N); r[g*T+t]=1; r[g*T+t-1]=-1; Aub.append(r); bub.append(md.ramp[g])
        Aub.append(-r); bub.append(md.ramp[g])
    res=linprog(c,A_ub=np.asarray(Aub) if Aub else None,b_ub=np.asarray(bub) if bub else None,
                A_eq=np.asarray(Aeq),b_eq=np.asarray(beq),bounds=bounds,method="highs")
    if not res.success: raise RuntimeError(res.message)
    p=res.x[pS].reshape(G,T); w=res.x[wS].reshape(3,T); flow=res.x[fS].reshape(3,T)
    lmp=-res.eqlin.marginals.reshape(3,T)
    return {"p":p,"w":w,"flow":flow,"lmp":lmp,"cost":float(res.fun),"res":res}


def clear_rt(md: MarketData, da, load: np.ndarray, wind_avail: np.ndarray,
             use_ramp=True, use_network=True):
    """RT redispatch around DA awards, including intertemporal ramping."""
    G,T=len(md.cost),24
    n=G*T; nw=3*T; nf=3*T; ns=3*T; N=2*n+nw+nf+ns
    up=slice(0,n); dn=slice(n,2*n); ws=slice(2*n,2*n+nw); fs=slice(ws.stop,ws.stop+nf); ss=slice(fs.stop,fs.stop+ns)
    c=np.zeros(N); c[up]=np.repeat(1.25*md.cost+2,T); c[dn]=np.repeat(-0.70*md.cost,T); c[ws]=-0.01; c[ss]=10000
    bounds=[]
    for g in range(G): bounds += [(0,max(0,md.pmax[g]-da['p'][g,t])) for t in range(T)]
    for g in range(G): bounds += [(0,max(0,da['p'][g,t])) for t in range(T)]
    for a in range(3): bounds += [(0,max(0,wind_avail[a,t])) for t in range(T)]
    fcaps=md.link_cap if use_network else np.full(3,1e5)
    for l in range(3): bounds += [(-fcaps[l],fcaps[l])]*T
    bounds += [(0,None)]*(3*T)
    Aeq=[]; beq=[]
    for a in range(3):
      for t in range(T):
        row=np.zeros(N); gs=np.where(md.gen_area==a+1)[0]
        row[gs*T+t]=1; row[dn.start+gs*T+t]=-1
        row[ws.start+a*T+t]=1; row[ss.start+a*T+t]=1
        for l,(u,v) in enumerate(LINKS):
            if a+1==u: row[fs.start+l*T+t]-=1
            if a+1==v: row[fs.start+l*T+t]+=1
        rhs=load[a,t]-da['p'][gs,t].sum()
        Aeq.append(row); beq.append(rhs)
    Aub=[];bub=[]
    if use_ramp:
      for g in range(G):
       for t in range(1,T):
        row=np.zeros(N)
        row[g*T+t]=1; row[g*T+t-1]=-1; row[dn.start+g*T+t]=-1; row[dn.start+g*T+t-1]=1
        base=da['p'][g,t]-da['p'][g,t-1]
        Aub.append(row); bub.append(md.ramp[g]-base)
        Aub.append(-row); bub.append(md.ramp[g]+base)
    res=linprog(c,A_ub=np.asarray(Aub) if Aub else None,b_ub=np.asarray(bub) if bub else None,
                A_eq=np.asarray(Aeq),b_eq=np.asarray(beq),bounds=bounds,method="highs")
    if not res.success: raise RuntimeError(res.message)
    lmp=-res.eqlin.marginals.reshape(3,T)
    return {"lmp":lmp,"cost":float(res.fun),"up":res.x[up].reshape(G,T),"dn":res.x[dn].reshape(G,T),"res":res}




## 4. White-box attacks

The profit- and cost-oriented attacks use central market sensitivities at every hour. A short line search handles price discontinuities caused by congestion/ramping active-set changes. This is a white-box vulnerability assessment: the attacker is assumed to know the forecast output and clearing model.

In [ ]:
def get_day(df, day_index):
    d=pd.to_datetime(df.timestamp).dt.normalize().drop_duplicates().iloc[day_index]
    x=df[pd.to_datetime(df.timestamp).dt.normalize().eq(d)].iloc[:24]
    def mat(prefix): return np.vstack([x[f"{prefix}_{a}"].to_numpy(float) for a in AREAS])
    return d, mat("load_da"), mat("load_rt"), mat("wind_da"), mat("wind_rt")


def evaluate(md, load_da, load_rt, wind_da, wind_rt, q, decompose=False):
    da=clear_da(md,load_da,wind_da); rt=clear_rt(md,da,load_rt,wind_rt)
    profit=float(np.sum(q*(da['lmp'][2]-rt['lmp'][2])))
    ans={"da":da,"rt":rt,"profit":profit,"system_cost":da['cost']+rt['cost']}
    if decompose:
        da_nr=clear_da(md,load_da,wind_da,use_ramp=False); rt_nr=clear_rt(md,da_nr,load_rt,wind_rt,use_ramp=False)
        da_e=clear_da(md,load_da,wind_da,use_ramp=False,use_network=False)
        rt_e=clear_rt(md,da_e,load_rt,wind_rt,use_ramp=False,use_network=False)
        E=q*(da_e['lmp'][2]-rt_e['lmp'][2])
        C=q*((da_nr['lmp'][2]-da_e['lmp'][2])-(rt_nr['lmp'][2]-rt_e['lmp'][2]))
        R=q*((da['lmp'][2]-da_nr['lmp'][2])-(rt['lmp'][2]-rt_nr['lmp'][2]))
        ans["pathways"]={"Energy":E,"Congestion":C,"Ramping":R}
    return ans


def coordinate_attack(md, ld, lr, wd, wr, q, eps, target="profit"):
    """White-box central-difference attack on area-3 forecast (24 coordinates)."""
    h=eps*md.wind_cap[2]
    grad=np.zeros(24)
    for t in range(24):
        vals=[]
        for sign in (-1,1):
            z=wd.copy(); z[2,t]=np.clip(z[2,t]+sign*h,0,md.wind_cap[2])
            ev=evaluate(md,ld,lr,z,wr,q)
            vals.append(ev["profit"] if target=="profit" else ev["system_cost"])
        grad[t]=(vals[1]-vals[0])/(2*h+1e-9)
    # A short white-box line search prevents simultaneous hourly moves from
    # cancelling one another after a market active-set transition.
    best=wd.copy()
    ev0=evaluate(md,ld,lr,wd,wr,q)
    best_value=ev0["profit"] if target=="profit" else ev0["system_cost"]
    for scale in (0.25,0.50,0.75,1.00):
        cand=wd.copy()
        cand[2]=np.clip(wd[2]+scale*h*np.sign(grad),0,md.wind_cap[2])
        ev=evaluate(md,ld,lr,cand,wr,q)
        value=ev["profit"] if target=="profit" else ev["system_cost"]
        if value>best_value+1e-8:
            best,best_value=cand,value
    return best,grad


def run_experiment(budget=0.02, train_days=30, test_days=7, verbose=True):
    data_path=DATA/"public_market_60d.csv"
    df=pd.read_csv(data_path) if data_path.exists() else build_public_dataset(60)
    md=load_market_data()
    # Ex-ante virtual position: sign of historical mean clean DA-RT spread by hour.
    spreads=[]
    for d in range(train_days):
        _,ld,lr,wd,wr=get_day(df,d); ev=evaluate(md,ld,lr,wd,wr,np.ones(24))
        spreads.append(ev['da']['lmp'][2]-ev['rt']['lmp'][2])
    q=100*np.sign(np.mean(spreads,axis=0)); q[q==0]=100
    rows=[]; detail={}
    for j,d in enumerate(range(train_days,train_days+test_days),1):
        date,ld,lr,wd,wr=get_day(df,d)
        clean=evaluate(md,ld,lr,wd,wr,q)
        h=budget*md.wind_cap[2]
        pgd=wd.copy(); pgd[2]=np.clip(wd[2]+h*np.sign(wd[2]-wr[2]),0,md.wind_cap[2])
        cost_adv,_=coordinate_attack(md,ld,lr,wd,wr,q,budget,"cost")
        profit_adv,g=coordinate_attack(md,ld,lr,wd,wr,q,budget,"profit")
        methods={"No attack":wd,"PGD-MSE":pgd,"Cost-oriented":cost_adv,"Profit-oriented":profit_adv}
        for name,z in methods.items():
            ev=clean if name=="No attack" else evaluate(md,ld,lr,z,wr,q)
            rows.append({"date":str(date.date()),"method":name,
                         "MSE":float(np.mean((z[2]-wr[2])**2)),
                         "system_cost":ev['system_cost'],"attacker_profit":ev['profit'],
                         "mean_abs_LMP_change":float(np.mean(np.abs(ev['da']['lmp'][2]-clean['da']['lmp'][2]))),
                         "profit_gain":ev['profit']-clean['profit'],"cost_change":ev['system_cost']-clean['system_cost']})
        detail[str(date.date())]={"q":q,"clean":clean,"profit_adv":evaluate(md,ld,lr,profit_adv,wr,q,True),
                                 "wind_clean":wd,"wind_actual":wr,"wind_adv":profit_adv,"gradient":g}
        if verbose: print(f"[{j}/{test_days}] {date.date()} completed")
    out=pd.DataFrame(rows); out.to_csv(RESULTS/"daily_results.csv",index=False)
    summary=out.groupby('method').agg(MSE=('MSE','mean'),system_cost=('system_cost','mean'),
        attacker_profit=('attacker_profit','mean'),profit_gain=('profit_gain','mean'),cost_change=('cost_change','mean'),
        mean_abs_LMP_change=('mean_abs_LMP_change','mean')).reset_index()
    base=summary.loc[summary.method.eq('No attack')].iloc[0]
    summary['Delta_MSE_pct']=100*(summary.MSE/base.MSE-1)
    summary['Delta_cost_pct']=100*summary.cost_change/base.system_cost
    summary.to_csv(RESULTS/"summary_results.csv",index=False)
    return summary,detail,md,df




## 5. Seven-day out-of-sample experiment

In [ ]:
# Set RUN_FULL=True to reproduce all market clears (about 8 minutes on a laptop).
RUN_FULL = True
if not (DATA/'public_market_60d.csv').exists():
    build_public_dataset(days=60)
if RUN_FULL:
    summary, detail, md_case, public_df = run_experiment(
        budget=0.02, train_days=30, test_days=7, verbose=True)
summary.round(3)


         method        MSE  system_cost  attacker_profit  profit_gain  cost_change  mean_abs_LMP_change  Delta_MSE_pct  Delta_cost_pct
  Cost-oriented 128500.384  1248837.241         1174.198     -976.335     4388.975                0.289          6.560           0.353
      No attack 120589.445  1244448.266         2150.534        0.000        0.000                0.000          0.000           0.000
        PGD-MSE 129156.465  1248822.985         1143.489    -1007.044     4374.719                0.302          7.104           0.352
Profit-oriented 119671.595  1244672.572         5437.995     3287.461      224.306                0.036         -0.761           0.018


The proposed attack raises mean private profit by **$3,287.461/day** while changing total system cost by only **0.018%**. PGD-MSE and the cost-oriented attack increase system cost by about 0.35% but reduce the attacker's profit on average. The proposed attack also reduces MSE by 0.761%, demonstrating that prediction-error monitoring can miss a profitable market attack.

## 6. IEEE one-column visualizations

In [ ]:
plt.rcParams.update({
    "font.family":"serif", "font.serif":["Times New Roman","Nimbus Roman"],
    "font.size":8, "axes.titlesize":8, "axes.labelsize":8,
    "xtick.labelsize":7, "ytick.labelsize":7, "legend.fontsize":7,
    "lines.linewidth":1.4, "axes.linewidth":0.8,
    "figure.dpi":180, "savefig.dpi":600,
})
COL={"PGD-MSE":"#377eb8","Cost-oriented":"#ff8c1a","Profit-oriented":"#d62728"}


def fig_outcomes():
    x=pd.read_csv(RESULTS/"daily_results.csv")
    methods=["PGD-MSE","Cost-oriented","Profit-oriented"]
    fig,ax=plt.subplots(1,2,figsize=(3.50,1.58),constrained_layout=True)
    vals=[x.loc[x.method.eq(m),"profit_gain"].to_numpy() for m in methods]
    bp=ax[0].boxplot(vals,positions=range(3),widths=.55,patch_artist=True,showfliers=False,
                     medianprops={"color":"black","linewidth":1})
    for box,m in zip(bp['boxes'],methods): box.set(facecolor=COL[m],alpha=.25,edgecolor=COL[m])
    for i,(v,m) in enumerate(zip(vals,methods)):
        jitter=np.linspace(-.13,.13,len(v)); ax[0].scatter(i+jitter,v,s=9,color=COL[m],zorder=3)
    ax[0].axhline(0,color='0.35',lw=.7); ax[0].set_xticks(range(3),["PGD","Cost","Profit"])
    ax[0].set_ylabel("Profit gain ($/day)"); ax[0].set_title("(a) Private payoff")

    for m in methods:
        d=x[x.method.eq(m)]
        ax[1].scatter(d.cost_change,d.profit_gain,s=10,color=COL[m],alpha=.65)
        ax[1].scatter(d.cost_change.mean(),d.profit_gain.mean(),s=35,color=COL[m],edgecolor='black',lw=.5,label=m)
    ax[1].axhline(0,color='0.35',lw=.7); ax[1].axvline(0,color='0.35',lw=.7)
    ax[1].set_xlabel("System-cost change ($/day)"); ax[1].set_ylabel("Profit gain ($/day)")
    ax[1].set_title("(b) Damage--profit decoupling")
    ax[1].legend(frameon=False,loc='upper right',handletextpad=.3,borderaxespad=.2)
    for a in ax: a.grid(True,color='0.9',lw=.5); a.tick_params(direction='in')
    for ext in ('pdf','png'): fig.savefig(FIGURES/f"fig1_market_outcomes.{ext}",bbox_inches='tight')
    plt.close(fig)


def _position(df,md):
    spreads=[]
    for d in range(30):
        _,ld,lr,wd,wr=get_day(df,d); ev=evaluate(md,ld,lr,wd,wr,np.ones(24))
        spreads.append(ev['da']['lmp'][2]-ev['rt']['lmp'][2])
    q=100*np.sign(np.mean(spreads,axis=0)); q[q==0]=100
    return q


def fig_pathways():
    df=pd.read_csv(DATA/"public_market_60d.csv"); md=load_market_data(); q=_position(df,md)
    daily=pd.read_csv(RESULTS/"daily_results.csv")
    rep=daily[daily.method.eq('Profit-oriented')].sort_values('profit_gain').iloc[-1].date
    days=pd.to_datetime(df.timestamp).dt.normalize().drop_duplicates().reset_index(drop=True)
    di=int(np.where(days.eq(pd.Timestamp(rep)))[0][0])
    _,ld,lr,wd,wr=get_day(df,di)
    adv,_=coordinate_attack(md,ld,lr,wd,wr,q,.02,'profit')
    clean=evaluate(md,ld,lr,wd,wr,q,True); attacked=evaluate(md,ld,lr,adv,wr,q,True)
    h=np.arange(1,25)
    fig,ax=plt.subplots(1,2,figsize=(3.50,1.58),constrained_layout=True)
    ax[0].plot(h,wr[2],color='0.65',label='Actual')
    ax[0].plot(h,wd[2],color='#377eb8',ls='--',label='Forecast')
    ax[0].plot(h,adv[2],color='#d62728',label='Profit attack')
    ax[0].set(xlabel='Hour',ylabel='Area-3 wind (MW)',title='(a) Forecast manipulation')
    ax[0].legend(frameon=False,ncol=1,loc='best',handlelength=2)
    for name,color in zip(['Energy','Congestion','Ramping'],['#d62728','#6a3d9a','#2ca02c']):
        delta=attacked['pathways'][name]-clean['pathways'][name]
        ax[1].plot(h,np.cumsum(delta),label=name,color=color)
    total=np.cumsum(sum(attacked['pathways'][k]-clean['pathways'][k] for k in attacked['pathways']))
    ax[1].plot(h,total,color='black',ls='--',label='Total')
    ax[1].axhline(0,color='0.4',lw=.7)
    ax[1].set(xlabel='Hour',ylabel='Cumulative profit gain ($)',title='(b) White-box pathways')
    ax[1].legend(frameon=False,loc='upper left',handlelength=1.8)
    for a in ax: a.grid(True,color='0.9',lw=.5); a.tick_params(direction='in'); a.set_xticks([1,6,12,18,24])
    for ext in ('pdf','png'): fig.savefig(FIGURES/f"fig2_whitebox_pathways.{ext}",bbox_inches='tight')
    plt.close(fig)
    pd.DataFrame({k:attacked['pathways'][k]-clean['pathways'][k] for k in attacked['pathways']}).assign(Total=lambda z:z.sum(axis=1)).to_csv(RESULTS/'representative_pathways.csv',index=False)



fig_outcomes()
fig_pathways()


![Market outcomes](figures/fig1_market_outcomes.png)

**Fig. 1. Private payoff and system-damage decoupling.**

![White-box pathways](figures/fig2_whitebox_pathways.png)

**Fig. 2. Forecast manipulation and market-price pathways.**

## 7. White-box completeness check

In [ ]:
pathway = pd.read_csv(RESULTS/'representative_pathways.csv')
print(pathway.sum().round(3))
assert abs(pathway[['Energy','Congestion','Ramping']].to_numpy().sum()
           - pathway['Total'].sum()) < 1e-6


Energy            0.000
Congestion    13166.479
Ramping           0.000
Total         13166.479


For the most profitable test day, the entire $13,166.479 gain is attributed to the **congestion-price pathway**, rather than energy or ramping scarcity. The three physical pathway contributions exactly reconstruct the total profit gain in the zonal clearing model. This provides a faithful, optimization-native explanation rather than a learned or post-hoc concept label.

## Outputs

- `results/daily_results.csv`: all day-method observations.
- `results/summary_results.csv`: paper-ready daily means.
- `results/representative_pathways.csv`: hourly physical profit attribution.
- `figures/*.pdf`: vector IEEE figures; PNG copies are included for preview.

**Research scope:** This is an offline academic benchmark. It does not connect to a live market or submit operational bids.